<a href="https://colab.research.google.com/github/Serragem/ModelagemVQE/blob/main/ModelahemH2%2C2Qb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit -U
!pip install qiskit_aer -U
!pip install qiskit-ibm-runtime -U

!pip install matplotlib
!pip install pylatexenc
!pip install qiskit-nature
!pip install pyscf

import qiskit
qiskit.__version__

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wh

'2.4.1'

In [ ]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit import transpile
from qiskit.visualization import plot_histogram, array_to_latex, plot_state_city
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize
import numpy as np
import matplotlib.pyplot as plt
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

## CRIAÇÃO DO DRIVER COM APOIO DA PySCFDriver

In [3]:
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import StatevectorEstimator
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper

distancia_H2 = 0.735

driver = PySCFDriver(
    atom=f"H 0 0 0; H 0 0 {distancia_H2}",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

problema_eletronico = driver.run()

## MAPEAMENTO FERMIÔNICO PARA QUIBITS

In [6]:
from qiskit_nature.second_q.mappers import ParityMapper
# aqui entra a chave para reduzir o numero de qubits

hamiltoniano_fermionico = problema_eletronico.hamiltonian.second_q_op()

mapper = ParityMapper(num_particles=problema_eletronico.num_particles)
# não entendi como funciona, quando fiz o código, deu problema aqui, mas aparentemente é uma função bem direta, não sei como funciona, mas sei que funciona

hamiltoniano_reduzido = mapper.map(hamiltoniano_fermionico)

print(f"Dimensionalidade após redução por simetria: {hamiltoniano_reduzido.num_qubits} qubits.")

Dimensionalidade após redução por simetria: 2 qubits.


## CONSTRUÇÃO DO ANSATZ COM TWOLOCAL

In [7]:
ansatz = TwoLocal(
    num_qubits=hamiltoniano_reduzido.num_qubits,
    rotation_blocks=['ry', 'rz'],
    entanglement_blocks='cz',
    reps=1
)
#no nosso caso só existe uma unica combinação de emaranhamento, portanto não é necessário declarar a "ordem" do emaranhamento
#entanglement entre quits
#Linear: O qubit 0 conecta-se ao 1; o 1 conecta-se ao 2; o 2 conecta-se ao 3, formando uma linha.
#Circular: Semelhante ao linear, mas o último qubit conecta-se de volta ao primeiro, fechando um anel.
#Total (Full): Todos os qubits conectam-se com todos os outros qubits da malha.

/tmp/ipykernel_3291/2551992330.py:1: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(


## CONFIGURAÇÃO DO ESTIMADOR E FUNÇÃO OBJETIVO

In [8]:
estimator = StatevectorEstimator()
ponto_inicial = np.zeros(ansatz.num_parameters)

def funcao_objetivo(parametros, ansatz_circ, hamiltoniano, estimador):
    job = estimador.run([(ansatz_circ, hamiltoniano, parametros)])
    resultado = job.result()[0]
    return resultado.data.evs

## OTIMIZAÇÃO CLASSICA VQE

In [12]:
print(f"Iniciando otimização com COBYLA para {ansatz.num_parameters} parâmetros...")

resultado_vqe = minimize(
    funcao_objetivo,
    ponto_inicial,
    args=(ansatz, hamiltoniano_reduzido, estimator),
    method='COBYLA',
    tol=1e-6
)

Iniciando otimização com COBYLA para 8 parâmetros...


## RESULTADOS

In [13]:
energia_eletronica = resultado_vqe.fun
energia_repulsao_nuclear = problema_eletronico.nuclear_repulsion_energy
energia_total = energia_eletronica + energia_repulsao_nuclear

print("\n" + "="*45)
print("RESULTADOS FINAIS DO VQE")
print("="*45)
print(f"Energia Eletrônica:          {energia_eletronica:.6f} Hartree")
print(f"Repulsão Nuclear:            {energia_repulsao_nuclear:.6f} Hartree")
print("-" * 45)
print(f"Energia Total do H2 (R={distancia_H2}Å): {energia_total:.6f} Hartree")
print("="*45)


RESULTADOS FINAIS DO VQE
Energia Eletrônica:          -1.857271 Hartree
Repulsão Nuclear:            0.719969 Hartree
---------------------------------------------
Energia Total do H2 (R=0.735Å): -1.137302 Hartree
